# Naples objective-and-result audit

## tl;dr

Retain descriptive objective/result components; this pilot does not support one common numerical grade. All four entries/eight sides remain represented. There are 19 distinct propositions, with eight explicit success-criterion nulls. The source reports intent before the relevant action for seven sides; Gaeta B also has a distinct later council purpose. Amantea B retains an inferred disposition.

This companion recomputes coverage and checks the partial/unknown cases. It fits no historical model and uses no reserved-final entries.


## Context & Methods

Scope: the complete frozen `third-coalition 1806` development group, including the December 1806-February 1807 Amantea interval. Grain: a source-qualified objective proposition within an entry side. Coverage always uses four entries/eight sides, rather than treating each proposition as another engagement.

### Key Assumptions

- A source's report of earlier intent is distinct from a supplied, dated pre-action document.
- Prescribed actions, intended effects, broader goals, expected consequences, commentary and inferred dispositions remain separate.
- An unknown criterion or result is not failure. Counts of heterogeneous propositions are not a score.
- A local capture does not establish regional pacification, redeployment or permanent force destruction.
- One source group's feasibility does not establish all-cohort feasibility or causal command contribution.


In [1]:
from pathlib import Path
from collections import Counter
import json
import sys

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "generalship/napoleonic_objectives.py").is_file())
sys.path.insert(0, str(project_root))
from generalship import napoleonic_objectives as audit
verified = audit.build(project_root)  # Fixed hashes, quotations and non-promotion.
source = verified["observations"]
rows = source["components"]
entry_ids = source["entry_ids"]
print("Verified pinned records; historical fits: 0; final entries used: 0")


Verified pinned records; historical fits: 0; final entries used: 0


## Data

Canonical observations, scope and protocol: `data/napoleonic/objective-pilot-v1/`.
Source-linked table and 39 exact passages: `napoleonic-naples-objective-pilot-v1.md`.
Saved reproducible profile: `artifacts/napoleonic-naples-objective-pilot-v1.json`.

The adopted evidence uses four existing author families, 26 inspected page images and six explicitly OCR-only passages. The eight old objective claims and 49 citation indices are retained; 27 prior-only pointers are not new verified propositions. Every raw source, dossier and earlier numerical record remains unchanged.


In [2]:
expected_keys = {(battle, side) for battle in entry_ids for side in ("A", "B")}
observed_keys = {(row["battle_id"], row["side"]) for row in rows}
assert len(entry_ids) == 4 and len(expected_keys) == 8
assert observed_keys == expected_keys
assert len(rows) == len({row["id"] for row in rows}) == 19

coverage = Counter()
print(f'{"Entry side":13} {"Reported intent timing":28} {"Dated prior document":20}')
for battle in entry_ids:
    for side in ("A", "B"):
        subset = [row for row in rows if (row["battle_id"], row["side"]) == (battle, side)]
        intent = [row for row in subset
                  if row["statement_kind"] not in ("retrospective_commentary", "inferred_disposition")]
        if any(row["reported_precedence"] == "before_relevant_action" for row in intent):
            timing = "before relevant action"
        elif any(row["reported_precedence"] == "before_later_phase" for row in intent):
            timing = "before later phase only"
        else:
            timing = "inferred disposition only"
        coverage[timing] += 1
        document = any(row["pre_action_document_evidence"] for row in subset)
        assert not document
        print(f'{battle + " / " + side:13} {timing:28} {str(document):20}')
assert tuple(coverage.values()) == (7, 1)
assert len({row["side"] for row in rows if row["battle_id"] == "B371a" and row["reported_precedence"] == "before_later_phase"}) == 1
print("Gaeta B: early refusal and later council remain separate propositions.")
print("Complete coverage: 4 entries / 8 sides; 19 unique proposition IDs; 0 dated pre-action objective documents adopted.")


Entry side    Reported intent timing       Dated prior document
B370c / A     before relevant action       False               
B370c / B     before relevant action       False               
B370d / A     before relevant action       False               
B370d / B     before relevant action       False               
B371a / A     before relevant action       False               
B371a / B     before relevant action       False               
B379c / A     before relevant action       False               
B379c / B     inferred disposition only    False               
Gaeta B: early refusal and later council remain separate propositions.
Complete coverage: 4 entries / 8 sides; 19 unique proposition IDs; 0 dated pre-action objective documents adopted.


## Results

### Criteria and reported relations

These exact-count tables audit the encoding. The relation vocabulary has no ordering or utility weights. Unknown or unassessable cases remain visible; neither is a zero or a failed mission.


In [3]:
missing_criteria = [row["id"] for row in rows if row["success_criterion"] is None]
assert len(missing_criteria) == 8
assert missing_criteria == verified["profile"]["missing_success_criteria"]
relations = dict(sorted(Counter(row["attainment"] for row in rows).items()))
assert relations == verified["profile"]["attainment_counts"]
print("Missing success criteria: 8 / 19 propositions")
for component in missing_criteria:
    print("  " + component)
print(f'\n{"Source-result relation (not a rating)":39} {"Propositions / 19":>18}')
for relation, count in relations.items():
    print(f'{relation:39} {count:>18}')
assert sum(relations.values()) == 19
assert source["common_grade"] is None and source["commander_credit"] is None
print("No numerical grade or commander-credit value is created.")


Missing success criteria: 8 / 19 propositions
  campo-calabria
  campo-counterfactual
  maida-insurrection
  gaeta-accelerate
  gaeta-early-defence
  gaeta-hold
  amantea-peace
  amantea-hold

Source-result relation (not a rating)    Propositions / 19
not_assessable                                           5
not_established                                          5
reported_action_executed                                 1
reported_effect_not_observed                             2
reported_effect_observed                                 2
reported_incomplete_or_mixed                             4
No numerical grade or commander-credit value is created.


### Material partial and unknown cases

The checks below prevent complete junction from replacing some joining, arrival from replacing a directional retreat, or a mission deadline from being manufactured from garbled OCR. They also keep a garrison's resistance separate from a documented prior objective. The full report retains the underlying quotations and alternative clocks.


In [4]:
indexed = {row["id"]: row for row in rows}
assert indexed["campo-junction"]["attainment"] == "reported_incomplete_or_mixed"
assert indexed["maida-catanzaro"]["attainment"] == "reported_incomplete_or_mixed"
assert indexed["gaeta-hold"]["criterion_literal"] == "dix4iait jours*"
assert indexed["gaeta-hold"]["normalized_duration_days"] is None
assert indexed["amantea-peace"]["attainment"] == "not_established"
assert indexed["amantea-hold"]["statement_kind"] == "inferred_disposition"
assert indexed["amantea-hold"]["success_criterion"] is None
assert indexed["maida-gaeta"]["related_entries"] == ["B371a"]
assert {row["id"] for row in rows if row["attainment"] == "reported_effect_observed"} == {"gaeta-capture", "amantea-capture"}
for component in ("campo-junction", "maida-catanzaro", "gaeta-hold", "amantea-peace", "amantea-hold"):
    print(component + ": " + indexed[component]["attainment"])
print("The two observed capture components remain local; no pacification or permanent force-destruction score follows.")
assert all(side["common_grade"] is None for side in verified["profile"]["side_coverage"])
assert verified["decision"]["historical_fits"] == verified["decision"]["reserved_final_entries_used"] == 0


campo-junction: reported_incomplete_or_mixed
maida-catanzaro: reported_incomplete_or_mixed
gaeta-hold: not_assessable
amantea-peace: not_established
amantea-hold: not_assessable
The two observed capture components remain local; no pacification or permanent force-destruction score follows.


## Takeaways

The component table adds inspectable distinctions beyond a recorded winner: prescribed action versus effect, partial joining, qualified retreat, capture, conditional release and unestablished strategic follow-through. The evidence does not supply a common graded-outcome scale for these different purposes.

Stop after this group. No new model input, historical run, wider source search or reopening of stopped appointment/opening research is authorized. All 52 final entries stay numerically reserved. This is primary verification and reproducible accounting, not an independent historical review.
